In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
from finagent.config.universe import universe
from finagent.market.features import compute_volume_change, flag_extreme_returns

In [ ]:
# Official universe: keep all 51 companies, including 99 Speed Mart.
assert len(universe) == 51
assert len({info["ticker"] for info in universe.values()}) == 51

In [ ]:
end_date = pd.to_datetime("2023-01-01")
end_date = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
start_date = end_date - pd.DateOffset(years=5)
print("Start:", start_date)
print("End:", end_date)

In [ ]:
def download_market_data(universe,start_date,end_date):
    all_data = []
    for company, info in universe.items():
        ticker = info["ticker"]
        print(f"Downloading {company} ({ticker})...")
        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=True,
            progress=False
        )
        if df.empty:
            print(f"No data for {ticker}")
            continue
        if isinstance(df.columns,pd.MultiIndex):
            df.columns = (df.columns.get_level_values(0))
        df = df.reset_index()
        df.columns = [col.lower() for col in df.columns]
        df["ticker"] = ticker
        df["company"] = company
        all_data.append(df)

    market_data = pd.concat(all_data,ignore_index=True)
    return market_data

In [ ]:
data = download_market_data(universe,start_date,end_date)
display(data.head())
print(data.shape)

In [ ]:
print(data.info())
print(data.isnull().sum())
print(data["company"].value_counts())

In [ ]:
data = data.drop_duplicates(subset=["date", "ticker"]).sort_values(["ticker", "date"]).reset_index(drop=True)
data["daily_return"] = data.groupby("ticker")["close"].pct_change(fill_method=None) * 100
data["intraday_return"] = (data["close"] - data["open"]) / data["open"] * 100
data["daily_range"] = (data["high"] - data["low"]) / data["open"] * 100
data["volume_change"] = compute_volume_change(data)
assert not np.isinf(data["volume_change"]).any()
data["direction"] = np.select([data["daily_return"] > 0, data["daily_return"] < 0], [1, -1], default=0)

market_quality_review = flag_extreme_returns(data)
if not market_quality_review.empty:
    print("UNRESOLVED DATA QUALITY ISSUE ? review these returns before training")
    display(market_quality_review)

In [ ]:
data = data.drop_duplicates(subset=["date","ticker"])
data = data.sort_values(["ticker","date"]).reset_index(drop=True)

In [ ]:
def add_technical_indicators(stock_df):
    stock_df = (stock_df.sort_values("date").copy())
    close = stock_df["close"]
    high = stock_df["high"]
    low = stock_df["low"]
    # ========================================================
    # SMA
    # ========================================================
    stock_df["sma_20"] = (close.rolling(20).mean())
    stock_df["sma_50"] = (close.rolling(50).mean())
    stock_df["sma_200"] = (close.rolling(200).mean())
    # ========================================================
    # EMA
    # ========================================================
    stock_df["ema_12"] = (close.ewm(span=12,adjust=False).mean())
    stock_df["ema_26"] = (close.ewm(span=26,adjust=False).mean())
    # ========================================================
    # MACD
    # ========================================================
    stock_df["macd"] = (stock_df["ema_12"]-stock_df["ema_26"])
    stock_df["macd_signal"] = (stock_df["macd"].ewm(span=9,adjust=False).mean())
    stock_df["macd_histogram"] = (stock_df["macd"]-stock_df["macd_signal"])
    # ========================================================
    # RSI 14
    # ========================================================
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta.clip(upper=0))
    avg_gain = gain.ewm(alpha=1 / 14,adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / 14,adjust=False).mean()
    rs = (avg_gain/avg_loss)
    stock_df["rsi_14"] = (100-(100/(1 + rs)))
    # ========================================================
    # ATR 14
    # ========================================================
    previous_close = close.shift(1)
    true_range = pd.concat([high - low,(high - previous_close).abs(),(low - previous_close).abs()],axis=1).max(axis=1)
    stock_df["atr_14"] = (true_range.ewm(alpha=1 / 14,adjust=False).mean())
    # ========================================================
    # BOLLINGER BANDS
    # ========================================================
    rolling_mean = (close.rolling(20).mean())
    rolling_std = (close.rolling(20).std())
    stock_df["bollinger_middle"] = (rolling_mean)
    stock_df["bollinger_upper"] = (rolling_mean + 2 * rolling_std)
    stock_df["bollinger_lower"] = (rolling_mean-2 * rolling_std)
    stock_df["bollinger_width"] = ((stock_df["bollinger_upper"]-stock_df["bollinger_lower"])/stock_df["bollinger_middle"]* 100)
    # ========================================================
    # VOLATILITY
    # ========================================================
    stock_df["volatility_20"] = (stock_df["daily_return"].rolling(20).std()*np.sqrt(252))
    # ========================================================
    # MOMENTUM
    # ========================================================
    stock_df["momentum_10"] = (close.pct_change(periods=10,fill_method=None)* 100)
    return stock_df

In [ ]:
feature_frames = []
for ticker, stock_df in data.groupby("ticker"):
    stock_features = (add_technical_indicators(stock_df))
    feature_frames.append(stock_features)
market_features = pd.concat(feature_frames,ignore_index=True)
market_features = market_features.sort_values(["ticker","date"]).reset_index(drop=True)

In [ ]:
display(market_features.tail())

In [ ]:
print(market_features.columns.tolist())

In [ ]:
latest_market_data = (market_features.sort_values("date").groupby("ticker",as_index=False).tail(1))
display(
    latest_market_data[
        [
            "date",
            "company",
            "ticker",
            "close",
            "daily_return",
            "rsi_14",
            "macd",
            "macd_signal",
            "volatility_20",
            "momentum_10"
        ]
    ])

In [ ]:
output_dir = Path("../data/preprocessed_data")
output_dir.mkdir(parents=True,exist_ok=True)
market_features.to_csv(output_dir/ "malaysia_market_features.csv",index=False)
print("Market features saved.")

In [ ]:
check_cols = [
    "date",
    "company",
    "ticker",
    "close",
    "daily_return",
    "rsi_14",
    "macd",
    "macd_signal",
    "atr_14",
    "volatility_20",
    "momentum_10"
]
latest = (market_features.sort_values("date").groupby("ticker").tail(1))
display(latest[check_cols])